# RAG & Knowledge Systems: Introduction to RAG

Welcome to Module 03 of the **Agentic AI Learning Path**. In this module, we explore **Retrieval-Augmented Generation (RAG)**—the architecture that allows AI agents to look up information from external datasets (PDFs, Databases, Web) to provide accurate, up-to-date, and context-aware answers.

## 1. What is RAG?
RAG is a technique that combines the generative power of LLMs with the precise retrieval of a search engine. 

### The Problem: Knowledge Cutoffs & Hallucinations
- LLMs have a training cutoff date (e.g., Gemini's internal knowledge might not know about today's news).
- LLMs can "hallucinate" facts when they don't know the answer.

### The Solution: The "Open Book" Exam
Instead of relying only on its internal knowledge (closed book), the agent searches for relevant documents (open book) and uses that information to answer.

---

## 2. The RAG Pipeline (ETL)
Building a RAG system involves four main steps:
1. **Load**: Importing data (PDF, Text, HTML).
2. **Split**: Breaking data into small "chunks."
3. **Embed**: Converting text chunks into mathematical vectors.
4. **Store**: Saving vectors in a "Vector Database" for fast lookup.

---

## 3. Environment Setup
We will use **LangChain**, **Google Gemini** (for reasoning), and **Hugging Face** (for local embeddings).

In [6]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Load environment variables
load_dotenv()

# Initialize Gemini for reasoning
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

print("Gemini & Environment initialized successfully!")

Gemini & Environment initialized successfully!


## 4. Understanding Vector Embeddings
Computers don't understand words; they understand numbers. **Embeddings** are representation of the "meaning" of a piece of text as a vector. We are using a local **Hugging Face** model here, which is free and fast.

In [7]:
model_name = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(model_name=model_name)

text1 = "I love programming in Python."
text2 = "Coding in Python is my favorite activity."
text3 = "The weather is cold today."

vector1 = embeddings.embed_query(text1)
vector2 = embeddings.embed_query(text2)
vector3 = embeddings.embed_query(text3)

print(f"Vector Length: {len(vector1)}")
print(f"First 5 values of Vector 1: {vector1[:5]}")

Vector Length: 384
First 5 values of Vector 1: [-0.057617057114839554, 0.00426226481795311, -0.028153114020824432, 0.0251975916326046, -0.016505613923072815]


## 5. Simulating a Knowledge Base
Let's create a tiny knowledge base of facts that the model wouldn't normally know.

In [8]:
knowledge_base = [
    "The Agentic AI Learning Path was created in 2024 to help developers build autonomous systems.",
    "The latest module in this path covers RAG and Vector Databases.",
    "Bala Murugan is the lead instructor for this series.",
    "The project uses Gemini 2.5 Flash as its primary generative engine."
]

## 6. Manual Retrieval (The Concept)
In a real system, a database does this. Here's what happens under the hood.

In [9]:
query = "Who is the lead instructor?"

def simple_retriever(query, documents):
    # We are simulating a search based on keywords
    results = [doc for doc in documents if "instructor" in doc.lower() or "bala" in doc.lower()]
    return "\n".join(results)

context = simple_retriever(query, knowledge_base)
print(f"Retrieved Context:\n{context}")

Retrieved Context:
Bala Murugan is the lead instructor for this series.


## 7. The Augmented Prompt
Now we pass the retrieved context along with the query to the LLM.

In [10]:
template = """
Answer the question based only on the following context:
{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)
chain = prompt | llm

response = chain.invoke({"context": context, "question": query})
print(f"AI Answer: {response.content}")

AI Answer: Bala Murugan


## 8. Automating with LangChain
LangChain's LCEL (LangChain Expression Language) allows us to build these pipelines effortlessly.

In [11]:
rag_chain = (
    {"context": lambda x: context, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print(rag_chain.invoke("What is the generative engine used?"))

Based on the provided context, there is no information about the generative engine used. The context only states that Bala Murugan is the lead instructor.


## 9. Challenges in RAG
RAG is powerful, but not perfect:
1. **Noise**: Retrieving irrelevant documents can confuse the model.
2. **Latency**: Searching through millions of vectors takes time.
3. **Context Window**: Even if you find 100 documents, they might not fit in the prompt.

---

## 10. Summary & Pro-Tips
1. **Embeddings are semantic**: Use them for meaning, not just exact words.
2. **Local vs Cloud**: Local embeddings (Hugging Face) are great for privacy and cost, while Cloud embeddings (Google) handle scale better.
3. **Next Steps**: In the next notebook, we will learn how to handle large files by splitting them into chunks.

---